In [0]:
# =========================
# MH ECDS Snapshots Pipeline (clean rebuild, 4 Parquets)
# - Stage 1: Prep, dates, helpers
# - Stage 2: Daily per-patient snapshot (one target day)
# - Stage 3: Aggregated Daily/Weekly/Monthly/Yearly
# - Stage 4: Export FOUR single-file Parquets for Power BI:
#       • Daily
#       • Weekly
#       • Monthly
#       • Yearly
# =========================

# ────────────────────────────────────────────────────────────────────────────────
# STAGE 1: PREP, DATES, HELPERS
# ────────────────────────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Safe/fast readers: disable DBIO cache (stale footer issues), keep vectorized parquet ON
spark.conf.set("spark.databricks.io.cache.enabled", "false")
spark.conf.set("spark.sql.sources.fileListingCacheEnabled", "false")
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
spark.conf.set("spark.sql.files.ignoreMissingFiles", "true")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.catalog.clearCache()

# Refresh any existing tables (best-effort)
for tbl in ["eds_snapshots.all_snapshots", "eds_snapshots.breaches_aggregated"]:
    try:
        spark.sql(f"REFRESH TABLE {tbl}")
        print(f"✅ Refreshed {tbl}")
    except Exception as e:
        print(f"ℹ️ Skipped refresh for {tbl}: {e}")

print("✅ Cache & metadata prepared")

# ─── IMPORTS ───────────────────────────────────────────────────────────────────
from pyspark.sql import functions as fn
from pyspark.sql.functions import (
    col, lit, to_date, row_number, broadcast, current_date, when, isnull,
    expr, coalesce, count, weekofyear, dayofweek, date_sub
)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, StringType
from pyspark.storagelevel import StorageLevel
from delta.tables import DeltaTable
from datetime import date, timedelta

# ─── PATHS ─────────────────────────────────────────────────────────────────────
daily_base = (
    "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
    "PATLondon/ECDS/Daily/pbi_snapshots_parquet/"
)

agg_summary_path = (
    "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
    "PATLondon/ECDS/Breaches_Aggregated/"
)

flat_agg_output = (
    "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
    "PATLondon/ECDS/Flat_Breaches_Aggregated/"
)

# ─── DATE SETTINGS ─────────────────────────────────────────────────────────────
backfill_from = date(2019, 1, 1)     # earliest snapshot date to ensure exists
lag_days     = 2                     # daily snapshot lag for completeness
today        = date.today()
StartDate    = today - timedelta(days=lag_days)   # last date we consider "stable"
StartDateStr = StartDate.isoformat()

DailyTarget     = StartDate          # per-patient daily snapshot target
DailyTargetStr  = DailyTarget.isoformat()

print(f"📅 Backfill from: {backfill_from} → up to (and incl.) {StartDate}")
print(f"📅 Daily per-patient snapshot target: {DailyTarget}")

# ─── HELPERS ───────────────────────────────────────────────────────────────────
def _path_exists(p: str) -> bool:
    try:
        dbutils.fs.ls(p)
        return True
    except Exception:
        return False

def ensure_dir(path: str):
    try:
        dbutils.fs.mkdirs(path.rstrip("/"))
    except Exception:
        pass

def ensure_delta_folder(path: str):
    if DeltaTable.isDeltaTable(spark, path):
        print(f"✅ Delta table present at {path}")
        return
    if _path_exists(path):
        print(f"⚠️ {path} exists but is NOT Delta; removing and recreating.")
        dbutils.fs.rm(path, recurse=True)
        ensure_dir(path)
    else:
        print(f"ℹ️ {path} does not exist; creating.")
        ensure_dir(path)

def week_start_monday(d_col):
    # Spark dayofweek: 1=Sun..7=Sat ⇒ offset to Monday = (dayofweek+5) % 7
    return date_sub(d_col, ((dayofweek(d_col) + 5) % 7))

def add_week_fields(df, date_col="snapshot_date"):
    return (
        df
        .withColumn("Week_Start_Mon", week_start_monday(col(date_col)))
        .withColumn("Week_Number", weekofyear(col(date_col)))
    )

def _non_empty(df):
    return df.limit(1).count() > 0

# ─── DETERMINE SNAPSHOT DATES TO BUILD (WITH OVERWRITE OF LAST 8 SNAPSHOT DAYS)
# ────────────────────────────────────────────────────────────────────────────────
# Full list of desired snapshot dates (calendar days) from backfill_from → StartDate
desired_dates = [
    backfill_from + timedelta(days=i)
    for i in range((StartDate - backfill_from).days + 1)
]

if DeltaTable.isDeltaTable(spark, agg_summary_path):
    agg_existing = (
        spark.read.format("delta")
        .load(agg_summary_path)
        .select("snapshot_date")
        .distinct()
    )

    # Python set of existing snapshot dates
    existing_set = {r.snapshot_date for r in agg_existing.collect()}

    # Latest snapshot_date that currently exists in the Delta
    max_snap = agg_existing.agg({"snapshot_date": "max"}).collect()[0][0]

    if max_snap is None:
        # Delta exists but has no rows yet
        todo_dates = desired_dates
    else:
        # Cap latest snapshot at StartDate (we only care up to the stable date)
        latest_snap = min(max_snap, StartDate)

        refresh_days = 8  # 👈 last N snapshot days to overwrite
        earliest_refresh = latest_snap - timedelta(days=refresh_days - 1)

        from pyspark.sql.functions import col as _col, lit as _lit

        dt = DeltaTable.forPath(spark, agg_summary_path)

        # Delete all rows in the refresh window, for all Periods
        dt.delete(
            (_col("snapshot_date") >= _lit(earliest_refresh))
            & (_col("snapshot_date") <= _lit(latest_snap))
        )
        print(
            f"🧹 Deleted existing aggregates for snapshot_date "
            f"{earliest_refresh} → {latest_snap} (all Periods)"
        )

        # Rebuild:
        #  - any historically missing dates
        #  - PLUS all dates in the refresh window
        missing_dates = [
            d for d in desired_dates
            if d not in existing_set and d <= StartDate
        ]
        refresh_dates = [
            d for d in desired_dates
            if earliest_refresh <= d <= latest_snap
        ]

        todo_dates = sorted(set(missing_dates).union(refresh_dates))

else:
    # Starting completely fresh – build everything from backfill_from → StartDate
    existing_set = set()
    todo_dates = desired_dates

print(f"⋙ Snapshot dates to build/refresh this run: {len(todo_dates)}")
if todo_dates[:5]:
    print(f"   e.g. {todo_dates[:5]}{' ...' if len(todo_dates) > 5 else ''}")

# For raw arrivals, look back one year before earliest needed snapshot_date
if todo_dates:
    earliest_needed = min(todo_dates)
else:
    earliest_needed = StartDate - timedelta(days=7)

raw_start = earliest_needed - timedelta(days=365)

print(f"⋙ Reading raw arrivals from: {raw_start} → < today()")

# ────────────────────────────────────────────────────────────────────────────────
# STAGE 2: READ RAW + REFERENCE; BUILD tempED2; DAILY PER-PATIENT SNAPSHOT
# ────────────────────────────────────────────────────────────────────────────────

# ─── READ RAW ──────────────────────────────────────────────────────────────────
raw_df = (
    spark.read.option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(
        "abfss://restricted@udalstdatacuratedprod.dfs.core.windows.net/"
        "patientlevel/MESH/ECDS/EC_Core/Published/1/"
    )
    .filter(col("Arrival_Date") >= lit(raw_start))
    .filter(col("Arrival_Date") < current_date())
    .filter(col("Deleted") == 0)
)

# ─── REFERENCE TABLES ──────────────────────────────────────────────────────────
trust_sites = (
    spark.read.format("delta")
    .option("header", "true").option("recursiveFileLookup", "true")
    .load(
        "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
        "PATLondon/MHUEC_Reference_Files/Trusts_and_Sites/"
    )
)


trusts_only = (
    spark.read.format("delta")
    .option("header", "true").option("recursiveFileLookup", "true")
    .load(
        "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
        "PATLondon/MHUEC_Reference_Files/Trusts_Only/"

    )
)

temp_gp7 = (
    spark.read.format("delta")
    .option("header", "true").option("recursiveFileLookup", "true")
    .load(
        "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
        "PATLondon/MHUEC_Reference_Files/GP_Data/"
    )
)

trust_boro_map = (
    spark.read.format("delta")
    .option("header", "true").option("recursiveFileLookup", "true")
    .load(
        "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
        "PATLondon/MHUEC_Reference_Files/Trust_Borough_Mapping/"
    )
)

treatment_function_code = (
    spark.read.option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(
        "abfss://unrestricted@udalstdatacuratedprod.dfs.core.windows.net/"
        "reference/Internal/Reference/TreatmentFunctionDetails/Published/"
    )
)

temp_gp6 = (
    spark.read.format("delta")
    .option("header", "true").option("recursiveFileLookup", "true")
    .load(
        "abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/"
        "PATLondon/MHUEC_Reference_Files/PostCode_to_LA/"
    )
)

ethnic_category_code = (
    spark.read.option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(
        "abfss://unrestricted@udalstdatacuratedprod.dfs.core.windows.net/"
        "reference/UKHD/Data_Dictionary/"
        "Ethnic_Category_Code_SCD/Published/1/"
        "UKHD_Data_Dictionary_Ethnic_Category_Code_SCD_00000.parquet"
    )
)

commissioner_hierarchies = (
    spark.read.option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(
        "abfss://reporting@udalstdatacuratedprod.dfs.core.windows.net/"
        "unrestricted/reference/UKHD/ODS/Commissioner_Hierarchies_ICB/"
    )
)

com_code_changes = (
    spark.read.option("header", "true")
    .option("recursiveFileLookup", "true")
    .parquet(
        "abfss://unrestricted@udalstdatacuratedprod.dfs.core.windows.net/"
        "reference/Internal/Reference/ComCodeChanges/Published/"
    )
)

distinct_site = trust_sites.select("Site_Organisation_Code", "Site_Name").distinct()

cc = broadcast(com_code_changes.alias("cc"))
o3 = broadcast(commissioner_hierarchies.alias("o3"))

# ─── CORE JOINS / FILTERS / DEDUPE ─────────────────────────────────────────────
df = (
    raw_df.join(
        cc,
        raw_df.Attendance_HES_CCG_From_Treatment_Site_Code == cc.Org_Code,
        "left"
    )
    .join(
        o3,
        coalesce(
            cc.New_Code,
            raw_df.Attendance_HES_CCG_From_Treatment_Site_Code
        ) == o3.Organisation_Code,
        "left"
    )
    .filter(
        (col("Deleted") == 0)
        & (col("EC_Department_Type") == "01")
        & (col("Arrival_Date") < current_date())
        & (
            col("EC_Discharge_Status_SNOMED_CT").isNull()
            | ~col("EC_Discharge_Status_SNOMED_CT").isin(
                ["1077031000000103", "1077781000000101", "63238001"]
            )
        )
        & (
            col("EC_AttendanceCategory").isin(["1", "2", "3"])
            | col("EC_AttendanceCategory").isNull()
        )
        & (coalesce(o3.Region_Name, lit("Missing/Invalid")) == "LONDON")
    )
    .withColumn(
        "Provider_Region_Code",
        coalesce(o3.Region_Code, lit("Missing/Invalid"))
    )
    .withColumn(
        "Provider_Region_Name",
        coalesce(o3.Region_Name, lit("Missing/Invalid"))
    )
    .withColumn(
        "Provider_CCGCode",
        coalesce(
            cc.New_Code,
            col("Attendance_HES_CCG_From_Treatment_Site_Code"),
            lit("Missing/Invalid"),
        )
    )
    .withColumn(
        "Provider_CCG_name",
        coalesce(o3.Organisation_Name, lit("Missing/Invalid"))
    )
    .withColumn(
        "Provider_STPCode",
        coalesce(o3.STP_Code, lit("Missing/Invalid"))
    )
    .withColumn(
        "Provider_STP_name",
        coalesce(o3.STP_Name, lit("Missing/Invalid"))
    )
)

w = Window.partitionBy(
    "Der_Pseudo_NHS_Number",
    "Attendance_Unique_Identifier",
    "Arrival_Date",
    "Der_EC_Departure_Date_Time",
    "Age_At_Arrival"
).orderBy("Arrival_Date")

df = df.withColumn("RowOrderECDS", row_number().over(w)).filter(
    col("RowOrderECDS") == 1
)
df = df.withColumn("ArrivalDate", to_date("Arrival_Date"))

# ─── SELECT CORE FIELDS (a) ────────────────────────────────────────────────────
a = df.select(
    "Generated_Record_ID", "Unique_CDS_identifier", "Attendance_Unique_Identifier",
    "Der_Pseudo_NHS_Number", "EC_Ident", "Sex",
    "Index_Of_Multiple_Deprivation_Decile",
    "Index_Of_Multiple_Deprivation_Decile_Description",
    "Accommodation_Status_SNOMED_CT", "Rural_Urban_Indicator", "Age_At_Arrival",
    "Ethnic_Category",
    lit(None).cast(FloatType()).alias(
        "Ethnic_proportion_per_100000_of_London_Borough_2020"
    ),
    lit(None).alias("Known_to_MH_Services_Flag"),
    lit(None).cast("date").alias("Last_Completed_IP_Spell"),
    lit(None).alias("IP_Spell_Provider_Name"),
    lit(None).alias("UniqHospProvSpellID"),
    lit(None).alias("IP_Spell_UniqServReqID"),
    lit(None).alias("ED_Presentation_within_28_days_of_Completed_IP_SPell"),
    lit(None).alias("Days_between_Completed_IP_Spell_and_ED_Presentation"),
    "PDS_General_Practice_Code", "GP_Practice_Code",
    "Attendance_Postcode_District",
    "Attendance_HES_CCG_From_Treatment_Origin",
    "Attendance_HES_CCG_From_Treatment_Site_Code",
    "Attendance_LSOA_Provider_Distance",
    "Attendance_LSOA_Treatment_Site_Distance",
    "Patient_Type",
    "Provider_Region_Code", "Provider_Region_Name",
    "Provider_CCGCode", "Provider_CCG_name",
    "Provider_STPCode", "Provider_STP_name",
    "Der_Provider_Code", "Provider_Code",
    "Der_Provider_Site_Code", "Site_Code_of_Treatment",
    "ArrivalDate", "Arrival_Time",
    "EC_Initial_Assessment_Date", "EC_Initial_Assessment_Time",
    "EC_Initial_Assessment_Time_Since_Arrival",
    "EC_Departure_Date", "EC_Departure_Time",
    "EC_Departure_Time_Since_Arrival",
    "EC_Seen_For_Treatment_Date", "EC_Seen_For_Treatment_Time",
    "EC_Seen_For_Treatment_Time_Since_Arrival",
    "EC_Conclusion_Date", "EC_Conclusion_Time",
    "EC_Conclusion_Time_Since_Arrival",
    "EC_Decision_To_Admit_Date", "EC_Decision_To_Admit_Time",
    "EC_Decision_To_Admit_Time_Since_Arrival",
    "Decision_To_Admit_Receiving_Site",
    "Decision_To_Admit_Treatment_Function_Code",
    "EC_Arrival_Mode_SNOMED_CT", "EC_Attendance_Source_SNOMED_CT",
    "Discharge_Follow_Up_SNOMED_CT", "Discharge_Destination_SNOMED_CT",
    "EC_Chief_Complaint_SNOMED_CT", "EC_Injury_Intent_SNOMED_CT",
    "Der_EC_Diagnosis_All",
    when(
        col("EC_Chief_Complaint_SNOMED_CT").isin(
            "248062006", "272022009", "48694002", "248020004",
            "6471006", "7011001", "366979004"
        ),
        1
    ).otherwise(0).alias("Chief_Complaint_Flag"),
    when(col("EC_Injury_Date").isNotNull(), 1).otherwise(0).alias("Injury_Flag"),
    when(col("EC_Injury_Intent_SNOMED_CT") == "276853009", 1)
    .otherwise(0).alias("Injury_Intent_Flag"),
    when(
        fn.coalesce(
            expr(
                """
                substring(
                    Der_EC_Diagnosis_All,
                    1,
                    case when instr(Der_EC_Diagnosis_All, ',') > 0
                         then instr(Der_EC_Diagnosis_All, ',') - 1
                         else length(Der_EC_Diagnosis_All)
                    end
                )
                """
            ),
            col("Der_EC_Diagnosis_All")
        ).isin(
            "52448006", "2776000", "33449004", "72366004", "197480006",
            "35489007", "13746004", "58214004", "69322001", "397923000",
            "30077003", "44376007", "17226007", "50705009"
        ),
        1
    ).otherwise(0).alias("Diagnosis_Flag"),
    when(
        col("EC_Chief_Complaint_SNOMED_CT").isin(
            "248062006", "272022009", "48694002", "248020004",
            "6471006", "7011001", "366979004"
        ),
        1
    )
    .when(col("EC_Injury_Intent_SNOMED_CT") == "276853009", 1)
    .when(
        fn.coalesce(
            expr(
                """
                substring(
                    Der_EC_Diagnosis_All,
                    1,
                    case when instr(Der_EC_Diagnosis_All, ',') > 0
                         then instr(Der_EC_Diagnosis_All, ',') - 1
                         else length(Der_EC_Diagnosis_All)
                    end
                )
                """
            ),
            col("Der_EC_Diagnosis_All")
        ).isin(
            "52448006", "2776000", "33449004", "72366004", "197480006",
            "35489007", "13746004", "58214004", "69322001", "397923000",
            "30077003", "44376007", "17226007", "50705009"
        ),
        1
    )
    .otherwise(0).alias("Mental_Health_Presentation_Flag"),
)

# ─── tempED (add TF desc/group) ────────────────────────────────────────────────
tf = treatment_function_code.alias("tf")
joined_tf = a.join(
    broadcast(tf),
    (col("Decision_To_Admit_Treatment_Function_Code") == col("tf.Main_Code_Text"))
    & (col("tf.Is_Latest") == lit(1)),
    "left",
)

tempED = (
    joined_tf
    .select(
        *[col(c) for c in a.columns],
        col("tf.Main_Description").alias("Treatment_Function_Desc"),
        col("tf.Category").alias("Treatment_Function_Group"),
    )
    .withColumn(
        "Gender",
        when(col("Sex") == "0", "Unknown")
        .when(col("Sex") == "1", "Male")
        .when(col("Sex") == "2", "Female")
        .when(col("Sex") == "9", "Not specified")
        .otherwise(None),
    )
    .withColumn(
        "Age_Group",
        when(
            (col("Age_At_Arrival") <= 18) & (~isnull(col("Age_At_Arrival"))),
            "CYP",
        )
        .when(
            (col("Age_At_Arrival") > 18) & (~isnull(col("Age_At_Arrival"))),
            "Adult",
        )
        .otherwise("Missing/Invalid"),
    )
    .withColumn(
        "AgeCat",
        when(
            (col("Age_At_Arrival") >= 0) & (col("Age_At_Arrival") <= 11),
            "0-11",
        )
        .when(
            (col("Age_At_Arrival") >= 12) & (col("Age_At_Arrival") <= 17),
            "12-17",
        )
        .when(
            (col("Age_At_Arrival") >= 18) & (col("Age_At_Arrival") <= 25),
            "18-25",
        )
        .when(
            (col("Age_At_Arrival") >= 26) & (col("Age_At_Arrival") <= 64),
            "26-64",
        )
        .when(col("Age_At_Arrival") >= 65, "65+")
        .otherwise("Missing/Invalid"),
    )
    .withColumn(
        "Time_Grouper",
        when(
            (col("EC_Departure_Time_Since_Arrival") >= 0)
            & (col("EC_Departure_Time_Since_Arrival") <= 240),
            "0-4",
        )
        .when(col("EC_Departure_Time_Since_Arrival").isNull(), "0-4")
        .when(
            (col("EC_Departure_Time_Since_Arrival") > 240)
            & (col("EC_Departure_Time_Since_Arrival") <= 720),
            "5-12",
        )
        .when(
            (col("EC_Departure_Time_Since_Arrival") > 720)
            & (col("EC_Departure_Time_Since_Arrival") <= 1440),
            "12-24",
        )
        .when(
            (col("EC_Departure_Time_Since_Arrival") > 1440)
            & (col("EC_Departure_Time_Since_Arrival") <= 2880),
            "24-48",
        )
        .when(
            (col("EC_Departure_Time_Since_Arrival") > 2880)
            & (col("EC_Departure_Time_Since_Arrival") <= 4320),
            "48-72",
        )
        .when(col("EC_Departure_Time_Since_Arrival") > 4320, ">72")
        .otherwise("Not recorded"),
    )
)

# ─── tempED2 (lookups) ─────────────────────────────────────────────────────────
o1 = trusts_only.alias("o1")
o2 = distinct_site.alias("o2")
gp = temp_gp7.alias("gp")
gpTm = trust_boro_map.alias("gpTm")
la = temp_gp6.alias("la")
ec = ethnic_category_code.alias("ec")

joined_lookups = (
    tempED
    .join(broadcast(o1), col("Der_Provider_Code") == o1.Parent_Organisation_Code, "left")
    .join(broadcast(o2), col("Site_Code_of_Treatment") == o2.Site_Organisation_Code, "left")
    .join(
        broadcast(gp),
        coalesce(col("PDS_General_Practice_Code"), col("GP_Practice_Code"))
        == gp.Practice_code,
        "left",
    )
    .join(broadcast(gpTm), gpTm.Borough == gp.Local_Authority_Name, "left")
    .join(broadcast(la), la.PCDS_NoGaps == o1.Parent_Organisation_Postcode, "left")
    .join(
        broadcast(ec),
        (ec.Main_Code_Text == col("Ethnic_Category"))
        & (ec.Is_Latest == lit(1)),
        "left",
    )
)

tempED2 = (
    joined_lookups
    .select(
        *[col(c) for c in tempED.columns],
        o1.Parent_Organisation_Name.alias("Parent_Organisation_Name"),
        o2.Site_Name.alias("Site_Name"),
        ec.Main_Description.alias("Derived_Broad_Ethnic_Category"),
        gpTm.Trust.alias("GP_Aligned_Trust"),
        gpTm.ICS.alias("GP_ICS"),
        gp.Local_Authority_Name.alias("GP_Local_Authority_Name"),
        col("Provider_Region_Name"),
    )
)

# ─── DAILY PER-PATIENT SNAPSHOT (ONE TARGET DAY) ───────────────────────────────
breaches_daily_df = (
    tempED2
    .select(
        col("ArrivalDate"),
        col("Parent_Organisation_Name").alias("ProviderName"),
        col("Site_Name").alias("ProviderSite"),
        when(
            col("Time_Grouper").isin("12-24", "24-48", "48-72", ">72"),
            lit(1),
        ).otherwise(lit(0)).alias("Breach12hr"),
        lit(None).cast("string").alias("Gap1"),
        lit(None).cast("string").alias("Gap2"),
        col("Age_Group").alias("AgeGroup"),
        col("Derived_Broad_Ethnic_Category"),
        lit("12 Plus Breach").alias("BreachFlag"),
        when(col("GP_Aligned_Trust").isNull(), "Others")
        .otherwise(col("GP_Aligned_Trust")).alias("MHTrust"),
        when(col("GP_ICS").isNull(), "OutOfLondonCCG_GPPracticeUnknown")
        .otherwise(col("GP_ICS")).alias("MHICS"),
        col("GP_Local_Authority_Name").alias("Borough"),
        lit(1).alias("DiagnosisLevel"),
        col("Unique_CDS_identifier"),
        col("EC_Ident"),
        when(col("GP_ICS").isNull(), "OutOfLondonCCG_GPPracticeUnknown")
        .otherwise("LondonCCG").alias("LondonCCG_OutOfLondonCCG"),
        when(col("Time_Grouper").isin("12-24", "24-48", "48-72", ">72"), 1)
        .otherwise(0).alias("Breach12hrs"),
        when(col("Time_Grouper") == "5-12", 1).otherwise(0).alias("Breach5to12hrs"),
        when(col("Time_Grouper") == "0-4", 1).otherwise(0).alias("Breach0to4hrs"),
        col("Time_Grouper").alias("TimeGrouper"),
        col("Mental_Health_Presentation_Flag").alias("MentalHealthPresentation"),
        col("Provider_Region_Name"),
        col("EC_Departure_Date"),
        col("EC_Departure_Time_Since_Arrival"),
    )
    .filter(
        (col("Site_Name").isNotNull())
        & (col("Site_Name") != "Missing/Invalid")
        & (col("Provider_Region_Name") == "LONDON")
        & (
            (col("ArrivalDate") == lit(DailyTarget))
            | (
                (col("ArrivalDate") < lit(DailyTarget))
                & col("EC_Departure_Date").between(lit(DailyTarget), lit(DailyTarget))
                & (col("EC_Departure_Time_Since_Arrival") >= 0)
            )
        )
    )
    .drop("EC_Departure_Date", "EC_Departure_Time_Since_Arrival")
    .withColumn("snapshot_date", lit(DailyTarget))
    .withColumn("Week_Start_Mon", week_start_monday(lit(DailyTarget)))
    .withColumn("Week_Number", weekofyear(lit(DailyTarget)))
)

daily_path = f"{daily_base}{DailyTargetStr}/"
ensure_dir(daily_base)

if breaches_daily_df.limit(1).count() == 0:
    print(f"⚠️ No records for ArrivalDate/DailyTarget={DailyTarget}. Writing empty snapshot.")
    spark.createDataFrame([], breaches_daily_df.schema).write.mode("overwrite").parquet(daily_path)
else:
    breaches_daily_df.write.mode("overwrite").parquet(daily_path)

print(f"📦 Daily per-patient snapshot written → {daily_path}")

# ────────────────────────────────────────────────────────────────────────────────
# STAGE 3: AGGREGATED DAILY / WEEKLY / MONTHLY / YEARLY
# ────────────────────────────────────────────────────────────────────────────────

analysis_df = (
    tempED2
    .select(
        col("ArrivalDate"),
        col("Parent_Organisation_Name").alias("Provider_Name"),
        col("Site_Name").alias("Provider_Site"),
        col("Age_Group").alias("Age_Group"),
        col("Derived_Broad_Ethnic_Category"),
        when(col("GP_Aligned_Trust").isNull(), "Others")
        .otherwise(col("GP_Aligned_Trust")).alias("Patient_Local_MH_Trust"),
        when(col("GP_ICS").isNull(), "OutOfLondonCCG_GPPracticeUnknown")
        .otherwise(col("GP_ICS")).alias("Patient_ICS"),
        col("GP_Local_Authority_Name").alias("Patient_Borough"),
        col("Mental_Health_Presentation_Flag").alias("Mental_Health_Presentation"),
        col("Time_Grouper"),
        col("Provider_Region_Name"),
        col("EC_Departure_Date"),
        col("EC_Departure_Time_Since_Arrival"),
    )
    .filter(
        (col("Site_Name").isNotNull())
        & (col("Site_Name") != "Missing/Invalid")
        & (col("Provider_Region_Name") == "LONDON")
    )
    .drop("Provider_Region_Name")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

_ = analysis_df.count()  # materialise

group_cols = [
    "Provider_Name", "Provider_Site", "Age_Group", "Derived_Broad_Ethnic_Category",
    "Patient_Local_MH_Trust", "Patient_ICS", "Patient_Borough",
    "Mental_Health_Presentation", "Time_Grouper",
]

def create_period_summary(df, start_dt, end_dt, label, snapshot_date=None):
    """
    start_dt / end_dt = attendance window
    snapshot_date     = label/date we stamp on the rows (defaults to end_dt)
    """
    if snapshot_date is None:
        snapshot_date = end_dt

    base = (
        df.filter(
            # (A) Arrivals in [start_dt, end_dt]
            ((col("ArrivalDate") >= lit(start_dt)) & (col("ArrivalDate") <= lit(end_dt)))
            |
            # (B) Arrivals before start, depart within window, dep time >= 0
            (
                (col("ArrivalDate") < lit(start_dt))
                & col("EC_Departure_Date").between(lit(start_dt), lit(end_dt))
                & (col("EC_Departure_Time_Since_Arrival") >= 0)
            )
        )
        .groupBy(*group_cols)
        .agg(count("*").alias("Total_Attendances"))
        .withColumn("Period", lit(label))
        .withColumn("snapshot_date", lit(snapshot_date))
    )
    return add_week_fields(base, "snapshot_date")

from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DateType

# Ensure destination exists as a Delta table
if not DeltaTable.isDeltaTable(spark, agg_summary_path):
    print(f"🆕 Creating new Delta table at {agg_summary_path}")
    empty_schema = StructType([
        StructField("Provider_Name", StringType()),
        StructField("Provider_Site", StringType()),
        StructField("Age_Group", StringType()),
        StructField("Derived_Broad_Ethnic_Category", StringType()),
        StructField("Patient_Local_MH_Trust", StringType()),
        StructField("Patient_ICS", StringType()),
        StructField("Patient_Borough", StringType()),
        StructField("Mental_Health_Presentation", IntegerType()),
        StructField("Time_Grouper", StringType()),
        StructField("Total_Attendances", LongType()),
        StructField("Period", StringType()),
        StructField("snapshot_date", DateType()),
        StructField("Week_Start_Mon", DateType()),
        StructField("Week_Number", IntegerType()),
    ])
    spark.createDataFrame([], empty_schema) \
        .write.format("delta").mode("overwrite").save(agg_summary_path)
else:
    print(f"✅ Delta table already exists at {agg_summary_path}")
wrote_any = False

for snap_dt in todo_dates:
    # ── WEEKLY WITH 1-WEEK LAG, MON–SUN WEEKS ─────────────────────────
    # snap_dt = run date
    # week_end = the SUNDAY we are REPORTING (one week before snap_dt)
    week_end = snap_dt - timedelta(days=7)       # e.g. 23rd Nov
    wk_start = week_end - timedelta(days=6)      # e.g. 17th Nov (Mon)

    # Month-end for the PREVIOUS month (we run monthly 7 days AFTER this date)
    prev_month_end = (snap_dt.replace(day=1) - timedelta(days=1))
    month_start    = prev_month_end.replace(day=1)
    month_end      = prev_month_end

    # 365-day trailing window ending at month_end (for Yearly)
    yr_start = month_end - timedelta(days=364)

    # Run weekly on SUNDAYS, and only once the week_end is within range
    # (weekday(): 0=Mon, 6=Sun)
    do_weekly  = (snap_dt.weekday() == 6) and (week_end >= backfill_from)

    # Run monthly & yearly exactly 7 days after previous month-end
    do_monthly = (snap_dt == prev_month_end + timedelta(days=7))
    do_yearly  = (snap_dt == prev_month_end + timedelta(days=7))

    # Always build DAILY – as-at snap_dt
    daily_summary = create_period_summary(
        analysis_df,
        start_dt=snap_dt,
        end_dt=snap_dt,
        label="Daily",
        snapshot_date=snap_dt
    )
    agg_batch = daily_summary

    if do_weekly:
        weekly_summary = create_period_summary(
            analysis_df,
            start_dt=wk_start,
            end_dt=week_end,        # Mon–Sun window
            label="Weekly",
            snapshot_date=week_end  # 🔴 label rows with the WEEK-END date
        )
        agg_batch = agg_batch.unionByName(weekly_summary)

    if do_monthly:
        monthly_summary = create_period_summary(
            analysis_df,
            start_dt=month_start,
            end_dt=month_end,
            label="Monthly",
            snapshot_date=month_end
        )
        agg_batch = agg_batch.unionByName(monthly_summary)

    if do_yearly:
        yearly_summary = create_period_summary(
            analysis_df,
            start_dt=yr_start,
            end_dt=month_end,
            label="Yearly",
            snapshot_date=month_end
        )
        agg_batch = agg_batch.unionByName(yearly_summary)

    # Diagnostic printout
    msg_parts = ["✓ Built aggregates for", str(snap_dt)]
    if do_weekly:  msg_parts.append("(weekly)")
    if do_monthly: msg_parts.append("(monthly)")
    if do_yearly:  msg_parts.append("(yearly)")
    print(" ".join(msg_parts))

    # ─── WRITE TO DELTA ───────────────────────────────────────────────
    if _non_empty(agg_batch):
        agg_batch = agg_batch.withColumn(
            "Total_Attendances",
            col("Total_Attendances").cast("long")
        )
        agg_batch.write \
            .format("delta") \
            .option("mergeSchema", "true") \
            .mode("append") \
            .save(agg_summary_path)
        wrote_any = True
        print(f"✅ Wrote aggregates for {snap_dt}")
    else:
        print(f"⚠️ No records to write for {snap_dt}")

analysis_df.unpersist()

# ────────────────────────────────────────────────────────────────────────────────
# STAGE 4: REGISTER TABLE & EXPORT FOUR ONE-FILE PARQUETS FOR PBI
# ────────────────────────────────────────────────────────────────────────────────

has_delta_now = DeltaTable.isDeltaTable(spark, agg_summary_path)

if has_delta_now:
    spark.sql("CREATE DATABASE IF NOT EXISTS eds_snapshots")
    ensure_dir(agg_summary_path)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS eds_snapshots.breaches_aggregated
        USING DELTA
        LOCATION '{agg_summary_path}'
    """)

    flat_df = spark.read.format("delta").load(agg_summary_path)

    flat_base = flat_agg_output.rstrip("/")

    def export_period(df, period_label, subfolder, filename):
        period_df = df.filter(col("Period") == lit(period_label))
        if not _non_empty(period_df):
            print(f"⚠️ No rows for Period='{period_label}'; skipping export.")
            return

        export_base = f"{flat_base}/{subfolder}"
        export_tmp  = f"{export_base}__tmp_{StartDateStr}"
        export_file = f"{export_base}/{filename}"

        ensure_dir(export_base)

        period_df.repartition(1).write.mode("overwrite") \
            .option("header", "true").parquet(export_tmp)

        try:
            dbutils.fs.rm(export_file, recurse=False)
        except Exception:
            pass

        parts = [f.path for f in dbutils.fs.ls(export_tmp)
                 if f.path.lower().endswith(".parquet")]
        if not parts:
            raise RuntimeError(f"No parquet file found in temp export folder: {export_tmp}")

        dbutils.fs.mv(parts[0], export_file)
        dbutils.fs.rm(export_tmp, recurse=True)

        print(f"✅ Exported {period_label} → {export_file}")

    export_period(flat_df, "Daily",   "Daily",   "breaches_aggregated_daily.parquet")
    export_period(flat_df, "Weekly",  "Weekly",  "breaches_aggregated_weekly.parquet")
    export_period(flat_df, "Monthly", "Monthly", "breaches_aggregated_monthly.parquet")
    export_period(flat_df, "Yearly",  "Yearly",  "breaches_aggregated_yearly.parquet")

    print("✅ Aggregated Delta updated/appended.")
    print(f"   • Delta path           : {agg_summary_path}")
    print(f"   • Flat PBI output base : {flat_base}")
    print(f"   • Daily snapshot       : {daily_path}")
else:
    print("⚠️ No aggregated data available yet at the Delta path; skipped registration and PBI export.")
    print(f"   • Checked path: {agg_summary_path}")
    print(f"   • Daily snapshot still written: {daily_path}")


ℹ️ Skipped refresh for eds_snapshots.all_snapshots: [DELTA_PATH_DOES_NOT_EXIST] abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/PATLondon/ECDS/AllSnapshotsDelta doesn't exist, or is not a Delta table.;
RefreshTable
+- ResolvedTable com.databricks.sql.managedcatalog.UnityCatalogV2Proxy@7739b96b, eds_snapshots.all_snapshots, DeltaTableV2(org.apache.spark.sql.SparkSession@5226db1b,abfss://analytics-projects@udalstdataanalysisprod.dfs.core.windows.net/PATLondon/ECDS/AllSnapshotsDelta,Some(CatalogTable(
Catalog: spark_catalog
Database: eds_snapshots
Table: all_snapshots
Owner: root
Created Time: Tue Jul 29 12:22:16 UTC 2025
Last Access: UNKNOWN
Created By: Spark 3.5.0
Type: EXTERNAL
Provider: delta
Table Properties: [delta.enableDeletionVectors=true, delta.feature.deletionVectors=supported, delta.lastCommitTimestamp=1753788580000, delta.lastUpdateVersion=0, delta.minReaderVersion=3, delta.minWriterVersion=7]
Location: abfss://analytics-projects@udalstdataanalysisprod.